# Importing Libraries

In [5]:
import pandas as pd
import numpy as np
from keplergl import KeplerGl
from pathlib import Path

In [2]:
Path = ("/Users/mehreenwerth/Desktop/citibike-weather-2022")

# Loading Files

In [3]:
trips = pd.read_csv("data_processed/citibike_master_2022.csv", low_memory=False)

weather = pd.read_csv("data_processed/weather_lga_2022.csv")

if "started_at" in trips.columns:
    trips["started_at"] = pd.to_datetime(trips["started_at"], errors="coerce")
weather["date"] = pd.to_datetime(weather["date"])

In [6]:
RAW_DIR = Path("data_raw/csv")

weather["date"] = pd.to_datetime(weather["date"])
weather["TAVG_C"] = weather["TAVG"] / 10

weather_small = weather[["date", "TAVG_C", "PRCP", "AWND"]].copy()
weather_small.head()

,date,TAVG_C,PRCP,AWND
0,2022-01-01,1.16,19.3,2.8
1,2022-01-02,1.14,1.0,4.3
2,2022-01-03,0.14,0.0,6.4
3,2022-01-04,-0.27,0.0,3.9
4,2022-01-05,0.32,6.1,3.4


In [7]:
weather["TAVG_C"] = weather["TAVG"] / 10

trips["date"] = trips["started_at"].dt.floor("D")

trips = trips.merge(weather[["date", "TAVG_C", "PRCP", "AWND"]], on="date", how="left")

trips[["date", "TAVG_C", "PRCP", "AWND"]].head()

,date,TAVG_C,PRCP,AWND
0,2022-01-21,-0.60,0.0,6.2
1,2022-01-10,0.16,0.0,7.5
2,2022-01-26,-0.23,0.0,5.6
3,2022-01-03,0.14,0.0,6.4
4,2022-01-22,-0.59,0.0,3.4


In [8]:
trips["trip_1"] = 1

start_name_col = "start_station_name"
end_name_col = "end_station_name"

print(start_name_col in trips.columns, end_name_col in trips.columns)

True False


In [9]:
files = sorted(list(RAW_DIR.glob("*.csv"))) + sorted(list(RAW_DIR.glob("*.csv.gz")))
len(files), files[:3]

(36,
 [PosixPath('data_raw/csv/202201-citibike-tripdata_1.csv'),
  PosixPath('data_raw/csv/202201-citibike-tripdata_2.csv'),
  PosixPath('data_raw/csv/202202-citibike-tripdata_1.csv')])

In [10]:
keep_cols = [
    "started_at", "ended_at",
    "start_station_name", "end_station_name",
    "start_lat", "start_lng", "end_lat", "end_lng",
    "member_casual"
]

parts = []
for f in files:
    temp = pd.read_csv(f, usecols=keep_cols, low_memory=False)
    parts.append(temp)

trips_map = pd.concat(parts, ignore_index=True)

trips_map["started_at"] = pd.to_datetime(trips_map["started_at"], errors="coerce")
trips_map["ended_at"] = pd.to_datetime(trips_map["ended_at"], errors="coerce")

trips_map["date"] = trips_map["started_at"].dt.floor("D")
trips_map = trips_map.merge(weather_small, on="date", how="left")

trips_map["duration_min"] = (trips_map["ended_at"] - trips_map["started_at"]).dt.total_seconds() / 60

trips_map.shape, trips_map.columns

((29838806, 14),
 Index(['started_at', 'ended_at', 'start_station_name', 'end_station_name',
        'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual', 'date',
        'TAVG_C', 'PRCP', 'AWND', 'duration_min'],
       dtype='object'))

# Aggregated Dataframe with Start Station, End Station, Count

In [11]:
trips_map["trip_1"] = 1

od_counts = (
    trips_map.dropna(subset=[
        "start_station_name","end_station_name",
        "start_lat","start_lng","end_lat","end_lng"
    ])
    .groupby([
        "start_station_name","end_station_name",
        "start_lat","start_lng","end_lat","end_lng"
    ])["trip_1"]
    .sum()
    .reset_index(name="trip_count")
)

od_counts.sort_values("trip_count", ascending=False).head(10)

,start_station_name,end_station_name,start_lat,start_lng,end_lat,end_lng,trip_count
1384775,Central Park S & 6 Ave,Central Park S & 6 Ave,40.765909,-73.976342,40.765909,-73.976342,10658
3713605,Roosevelt Island Tramway,Roosevelt Island Tramway,40.757284,-73.953600,40.757284,-73.953600,7862
628809,7 Ave & Central Park South,7 Ave & Central Park South,40.766741,-73.979069,40.766741,-73.979069,7378
3797938,Soissons Landing,Soissons Landing,40.692317,-74.014866,40.692317,-74.014866,6731
4265192,W 21 St & 6 Ave,9 Ave & W 22 St,40.741740,-73.994156,40.745497,-74.001971,5987
2688229,Grand Army Plaza & Central Park S,Grand Army Plaza & Central Park S,40.764397,-73.973715,40.764397,-73.973715,5858
41383,1 Ave & E 62 St,1 Ave & E 68 St,40.761227,-73.960940,40.765005,-73.958185,5479
499722,5 Ave & E 72 St,5 Ave & E 72 St,40.772828,-73.966853,40.772828,-73.966853,5318
146558,12 Ave & W 40 St,12 Ave & W 40 St,40.760875,-74.002777,40.760875,-74.002777,4979
5004586,Yankee Ferry Terminal,Yankee Ferry Terminal,40.687066,-74.016756,40.687066,-74.016756,4846


# Kepler Map

In [12]:
od_counts.shape

(5004655, 7)

In [13]:
TOP_N = 2000

od_top = (
    od_counts.sort_values("trip_count", ascending=False)
             .head(TOP_N)
             .copy()
)

od_top.shape

(2000, 7)

In [14]:
from keplergl import KeplerGl

map_1 = KeplerGl(height=650)
map_1.add_data(data=od_top, name="OD Trips")
map_1

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(data={'OD Trips': {'index': [1384775, 3713605, 628809, 3797938, 4265192, 2688229, 41383, 499722, 1465…

## Configuring

In [16]:
config = {
  "version": "v1",
  "config": {
    "visState": {
      "filters": [],
      "layers": [
        {
          "id": "start_points",
          "type": "point",
          "config": {
            "dataId": "OD Trips",
            "label": "Start Stations",
            "columns": {"lat": "start_lat", "lng": "start_lng"},
            "isVisible": True,
            "visConfig": {
              "radius": 8,
              "opacity": 0.7
            }
          },
          "visualChannels": {
            "sizeField": {"name": "trip_count", "type": "integer"},
            "sizeScale": "sqrt"
          }
        },
        {
          "id": "od_arcs",
          "type": "arc",
          "config": {
            "dataId": "OD Trips",
            "label": "OD Arcs",
            "columns": {
              "lat0": "start_lat",
              "lng0": "start_lng",
              "lat1": "end_lat",
              "lng1": "end_lng"
            },
            "isVisible": True,
            "visConfig": {
              "opacity": 0.35,
              "thickness": 2
            }
          },
          "visualChannels": {
            "sizeField": {"name": "trip_count", "type": "integer"},
            "sizeScale": "sqrt"
          }
        }
      ]
    }
  }
}

In [17]:
map_1.config = config
map_1

KeplerGl(config={'version': 'v1', 'config': {'visState': {'filters': [], 'layers': [{'id': 'start_points', 'ty…

## Filter Requirement

In [18]:
upper = int(od_top["trip_count"].quantile(0.80))
config["config"]["visState"]["filters"] = [{
    "dataId": ["OD Trips"],
    "id": "tripcount_filter",
    "name": ["trip_count"],
    "type": "range",
    "value": [upper, int(od_top["trip_count"].max())],
    "enlarged": True
}]

map_1.config = config
map_1

KeplerGl(config={'version': 'v1', 'config': {'visState': {'filters': [{'dataId': ['OD Trips'], 'id': 'tripcoun…

# NYC 'Busy Zones'

In [19]:
start_station_counts = (
    od_counts.groupby(["start_station_name", "start_lat", "start_lng"], as_index=False)["trip_count"]
             .sum()
             .sort_values("trip_count", ascending=False)
             .head(2000)
)

map_2 = KeplerGl(height=650)
map_2.add_data(start_station_counts, name="Start Stations")
map_2

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(data={'Start Stations': {'index': [3420466, 3957585, 987102, 456281, 850206, 49349, 3215772, 915301, …